# Chapter 7: Search In-Depth

## Topic 3: Advanced Retrieval — DPR, ColBERT, SPLADE, and Learned Sparse Retrieval

---

### 1. Concept and Intuition

**Why this topic exists — the limitations of what you have built so far:**

- Topic 1 (BM25): exact term matching, fast, free, zero when vocabulary does not overlap
- Topic 2 (dense retrieval with `paraphrase-multilingual-MiniLM-L12-v2`): semantic matching, bridges vocabulary gaps, but the measured discrimination gap for your corpus is only 0.0393 — dangerously thin for 31-word Hinglish emails
- Both methods treat query and document as single monolithic vectors and compare them in a single step after the fact

The architectures in this topic address different failure modes by changing the fundamental assumption:

- **DPR (Dense Passage Retrieval, 2020, Meta)**: uses two separate encoders — one trained specifically on queries, one trained specifically on passages — instead of one symmetric encoder used for both. The key insight is that questions and answers have different linguistic structures; the same encoder is not optimal for both.
- **ColBERT (Contextualized Late Interaction over BERT, 2020, Stanford)**: instead of compressing an entire document into one vector, keeps one vector per token. Matching happens token-by-token between query and document — "late interaction" — giving much richer comparison at the cost of more storage and computation.
- **SPLADE (SParse Lexical AnD Expansion, 2021, NAVER Labs)**: uses a transformer to expand each document's sparse representation into a learned vocabulary — generating implicit terms the document never states but implies. This is learned sparse retrieval — BM25's speed and interpretability combined with a transformer's semantic breadth.
- **Learned sparse retrieval (general)**: the family of approaches that use transformer models to produce sparse vectors over the full vocabulary, where non-zero dimensions correspond to actual vocabulary terms with learned weights, rather than raw TF counts.

**The progression from what you know:**

```
BM25               → exact term overlap, weighted by corpus statistics
Dense (bi-encoder) → single vector per text, cosine similarity
DPR                → asymmetric bi-encoder: separate query and document towers
ColBERT            → multi-vector per text, MaxSim late interaction
SPLADE             → learned sparse expansion, best of both BM25 and dense
```

---

### 2. DPR — Dense Passage Retrieval

#### 2.1 What It Is

- DPR (Karpukhin et al., 2020, "Dense Passage Retrieval for Open-Domain Question Answering", Meta AI)
- Core idea: train two separate BERT-based encoders from scratch on (question, positive passage, negative passage) triplets
- Query encoder `E_Q`: takes the user's question and produces a 768-dim vector
- Passage encoder `E_P`: takes a document chunk and produces a 768-dim vector
- At inference: embed all passages with `E_P` once, store in FAISS; embed each query with `E_Q` at query time, retrieve nearest passages

#### 2.2 Why Two Encoders Instead of One

- Questions and answers have fundamentally different linguistic structures
- A question: "What is the penalty for premature FD withdrawal?" — short, interrogative, contains "what", question marks, sometimes incomplete
- An answer/passage: "Premature withdrawal incurs a 1 percent penalty on the applicable interest rate" — declarative, complete, no question words
- A single symmetric encoder (what `paraphrase-multilingual-MiniLM-L12-v2` is) learns a compromise that is suboptimal for both
- DPR's two encoders each specialise: `E_Q` learns what question embeddings should look like, `E_P` learns what answer embeddings should look like, and training pulls matched pairs together while pushing non-matches apart

#### 2.3 Training Objective

- Contrastive learning with hard negatives
- For each (question, positive passage) pair, the model sees several negative passages — documents that look relevant but are not the correct answer
- Hard negatives: passages retrieved by BM25 that look like good matches but are not the gold answer — forcing the model to learn fine-grained distinctions
- Loss function: maximise similarity of `(E_Q(question), E_P(positive_passage))` while minimising similarity of `(E_Q(question), E_P(negative_passage))` — in-batch negatives plus hard negatives

#### 2.4 How DPR Relates to Your Project

- Your current `paraphrase-multilingual-MiniLM-L12-v2` is a symmetric bi-encoder — the same model produces embeddings for both queries (customer emails) and passages (policy document chunks)
- This is a DPR-style architecture but without the asymmetric specialisation — the model was not specifically trained on (FD question, FD answer) pairs
- A proper DPR model fine-tuned on your corpus would use customer emails as queries and policy chunk answers as passages
- The `E5` model family ("intfloat/multilingual-e5-large") is the modern practical version of this idea — explicitly trained with "query:" and "passage:" prefixes to distinguish the two roles, available on Hugging Face

#### 2.5 Strengths and Weaknesses

**Strengths:**
- Significantly outperforms symmetric bi-encoders on retrieval benchmarks where query and passage styles differ significantly
- Both encoders are BERT-based: fast at inference (single forward pass per text), scales to millions of passages with FAISS
- Can be fine-tuned on domain-specific (question, passage) pairs — e.g. your FD email / policy chunk pairs

**Weaknesses:**
- Requires labeled (question, positive passage) training pairs — expensive to create without a labeled dataset
- Still a single-vector-per-text representation: loses fine-grained token-level matching that ColBERT provides
- Vocabulary mismatch is reduced but not eliminated — if the question contains "byaj" and the passage encoder has never seen it in a financial context, the mapping is still imprecise

---

### 3. ColBERT — Contextualized Late Interaction

#### 3.1 What It Is

- ColBERT (Khattab & Zaharia, 2020, "ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction over BERT", Stanford)
- Core idea: instead of producing one vector per document, produce one vector per token. Scoring is done by comparing every query token against every document token.
- Architecture: BERT encoder for both query and document (can be the same encoder or separate)
- Each token in the query produces a vector; each token in the document produces a vector
- Final relevance score = `MaxSim`: for each query token, find the maximum cosine similarity with any document token, then sum those maxima across all query tokens

#### 3.2 The MaxSim Scoring Function

```text
ColBERT score(q, d) = Σ  max  cos_sim(E_q(q_i), E_d(d_j))
                       i∈q  j∈d

where:
  q_i = the i-th token in the query
  d_j = the j-th token in the document
  E_q, E_d = query and document encoders (may be shared)
```

**What this means in practice:**

For the query "premature withdrawal penalty":
- Token "premature" finds its best match in the document: maybe the document token "premature" (direct match) or "early" (semantic match) — whichever is closer in vector space
- Token "withdrawal" finds its best match: maybe "withdrawal" or "closure" or "exit"
- Token "penalty" finds its best match: maybe "penalty" or "charges" or "fee"
- The final score is the sum of these three best matches

**Why this is much more expressive than single-vector retrieval:**
- A single vector for "premature withdrawal penalty" is a blended average of all three concepts — it loses the fact that each concept independently needs to match somewhere in the document
- MaxSim checks each concept independently — "penalty" must match somewhere, "premature" must match somewhere, "withdrawal" must match somewhere — and each contributes to the final score independently

#### 3.3 Why ColBERT Is Expensive

- A document with 100 tokens produces 100 vectors × 128 dimensions = 12,800 numbers per document
- Compared to a single 384-dimensional vector in your current approach: 100× more storage per document
- At query time, you must compare every query token against every document token — cannot use a simple FAISS nearest-neighbor search directly
- ColBERT addresses this with a two-stage approach: coarse candidate retrieval using compressed representations, then exact MaxSim scoring on the candidates

#### 3.4 ColBERT v2 Improvements

- Residual compression: stores compressed token vectors that can be decompressed for exact MaxSim scoring
- Denoised supervision: uses a more sophisticated negative mining strategy
- Achieves 4–6× storage reduction over original ColBERT while maintaining quality
- `ragatouille` library provides a practical ColBERT v2 implementation ready to use

#### 3.5 How ColBERT Relates to Your Project

- Your corpus is small (17 pages) — ColBERT's storage overhead is completely manageable at this scale
- The specific failure mode ColBERT helps with: "premature withdrawal" in a query matching a document that says "early closure penalty" — ColBERT would match "withdrawal" → "closure" and "premature" → "early" as separate token-level matches, while your current model blends them into one vector that may or may not be close
- ColBERT is overkill at 17 pages but becomes compelling at tens of thousands of chunks where single-vector retrieval's discrimination gap creates measurable answer quality problems

---

### 4. SPLADE — Learned Sparse Retrieval

#### 4.1 What It Is

- SPLADE (Formal et al., 2021, "SPLADE: Sparse Lexical and Expansion Model for First Stage Retrieval", NAVER Labs Europe)
- Core idea: use a BERT-based MLM (Masked Language Model) head to produce a sparse vector over the full vocabulary where non-zero dimensions correspond to actual vocabulary terms with learned weights
- The model learns to "expand" documents — generating implicit terms that the document implies but never states

#### 4.2 How SPLADE Works

**At document encoding time:**
1. Pass the document through BERT
2. Apply the MLM head: for each token position, compute scores over the entire vocabulary (30,000+ terms)
3. Apply ReLU to keep only positive scores (sparsity)
4. Apply max-pooling over all token positions to aggregate the vocabulary scores
5. Apply log(1 + score) transformation for further sparsity
6. Result: a sparse vector with one dimension per vocabulary term — most are zero, a few hundred are non-zero

**What this produces:**
- A policy chunk saying "premature withdrawal incurs a penalty" might produce a sparse vector with high weights for: "premature", "withdrawal", "penalty", "FD", "closure", "charges", "forfeit", "deduction" — terms implied by the context but not all explicitly stated
- A customer email saying "mera FD band karna hai" might produce a sparse vector with weights for: "FD", "band", "karna", "close", "closure", "exit", "premature", "withdrawal" — bridging the Hinglish and English vocabulary

#### 4.3 Why SPLADE Beats BM25

- BM25: only matches terms that literally appear in the document — zero expansion
- SPLADE: the transformer learns during training which terms co-occur in similar contexts and expands accordingly — implicit vocabulary bridging
- For your Hinglish corpus: SPLADE might learn that "byaj" and "interest" appear in similar contexts and give "byaj" a non-zero weight in documents about interest rates, even when "byaj" never appears in those documents
- In benchmarks: SPLADE significantly outperforms BM25 while remaining interpretable (you can see which terms have non-zero weights and understand why)

#### 4.4 Why SPLADE Is Attractive for Your Project Specifically

- Sparse representation: can use standard inverted indexes (same infrastructure as BM25, Elasticsearch) — no FAISS, no Qdrant vector search needed
- Interpretable: the non-zero vocabulary terms tell you exactly why a document was retrieved
- Bridges vocabulary mismatch: the key problem in your 64.4% Hinglish corpus
- No custom tokenizer needed: works with standard WordPiece tokenizer
- Available via the `splade` library from NAVER Labs

#### 4.5 SPLADE Variants

- **SPLADE**: original model, both query and document expanded
- **SPLADE-max**: uses max pooling (as described above)
- **SPLADE-v2**: improved training with hard negatives and distillation from a cross-encoder
- **SPLADE++**: further improved with ensemble distillation
- **uniCOIL**: lighter variant, produces one weight per token (not over full vocabulary) — faster but less expressive

---

### 5. The Broader Landscape of Learned Sparse Retrieval

#### 5.1 What "Learned Sparse" Means

Traditional sparse (BM25) → term weights from corpus statistics (TF-IDF math)
Learned sparse (SPLADE, uniCOIL) → term weights from transformer model trained on relevance signal

The key distinction: the weights are no longer derived from raw frequency counts but from a model that has learned what terms matter for relevance in context.

#### 5.2 uniCOIL — A Simpler Alternative

- uniCOIL (Lin & Ma, 2021): instead of expanding over the full vocabulary, produces one weight per input token
- Much simpler: each token in the document gets a scalar weight (learned, not TF-based)
- Combined with query expansion (adding relevant terms to the query using a separate model), achieves strong retrieval performance
- Much smaller model, faster inference than full SPLADE

#### 5.3 PLAID — ColBERT at Scale

- PLAID (Santhanam et al., 2022): an optimised ColBERT inference engine
- Uses centroid-based compressed representations to dramatically speed up the first-stage candidate retrieval
- Makes ColBERT practically deployable at the scale of Wikipedia (21 million passages)
- Available via the `ragatouille` library

#### 5.4 Where Each Fits in the Cost-Quality Ladder

From cheapest and least accurate to most expensive and most accurate:

- BM25 — milliseconds, no GPU, lowest quality on vocabulary mismatch
- Dense bi-encoder (current project) — 20–50ms CPU embed, moderate quality
- DPR / asymmetric bi-encoder — same latency as bi-encoder, better quality when query and passage styles differ
- uniCOIL / SPLADE — GPU needed at index time (~seconds per document), millisecond BM25-style search at query time, good quality
- ColBERT — more storage, multi-stage retrieval, high quality
- Cross-encoder (reranker, Topic 7) — very slow per candidate, highest quality, used only on top-k candidates not full corpus

---

### 6. How These Relate to This Project

**Current state:**
- BM25 (Topic 1): implemented, works for exact term matches in English
- Dense bi-encoder (Topic 2): implemented via `paraphrase-multilingual-MiniLM-L12-v2`, gap = 0.0393

**The upgrade path:**

- **Short term — use E5 instead of paraphrase-MiniLM:** `intfloat/multilingual-e5-large` is an asymmetric model that uses "query:" and "passage:" prefixes, closer to DPR in spirit, available as a drop-in replacement, significantly better than symmetric models on retrieval benchmarks. No training needed.

- **Medium term — SPLADE for your Hinglish problem:** SPLADE's vocabulary expansion is specifically what bridges "byaj" (Hinglish) to "interest" (English) and "machurity" to "maturity". If fine-tuned on (Hinglish email, English policy chunk) pairs, SPLADE would learn these mappings explicitly. Available via the `splade` library.

- **Production scale — hybrid of BM25 + dense with RRF (Topic 4):** the fastest path to quality improvement without any training. BM25 handles exact matches and novel product names. Dense handles vocabulary mismatch. RRF merges the ranked lists. Topic 4 implements this in full.

- **High quality baseline — ColBERT v2 via ragatouille:** at 17 pages, ColBERT v2 is completely feasible. `ragatouille` provides a 4-line setup. Token-level matching would resolve the "withdrawal" / "closure" / "exit" vocabulary variation that BM25 and single-vector retrieval both handle imprecisely.

---

### 7. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

**DPR:**
- Requires labeled (question, passage) training data — the main barrier for domain-specific use
- Hard negatives dramatically improve quality but are expensive to mine well — BM25 hard negatives are the standard starting point
- At scale: both `E_Q` and `E_P` run as separate inference services; FAISS indexes `E_P` embeddings; query time uses `E_Q` to embed then FAISS to retrieve
- Latency: same as standard bi-encoder dense retrieval (~20–50ms CPU, ~2–5ms GPU for embedding)

**ColBERT:**
- Storage: 100 tokens × 128 dims × 4 bytes = ~50KB per document vs ~1.5KB for single-vector at 384 dims
- At 17 pages × ~250 tokens/page ≈ 4,250 token vectors = trivial
- At 100,000 pages × 250 tokens ≈ 25 million token vectors = ~3.2GB — manageable on modern hardware
- Query time: two-stage — fast first-pass candidate retrieval (compressed), then exact MaxSim reranking
- `ragatouille` handles all of this automatically

**SPLADE:**
- Index time: GPU required (transformer inference per document), typically 1–10 seconds per batch of documents — fine for offline indexing, not for real-time single-document indexing
- Query time: millisecond inverted index lookup (same infrastructure as BM25) — no GPU needed at query time
- The expanded sparse vector for a 20-word document might have 200–500 non-zero dimensions — still sparse enough for efficient inverted index operations
- Storage: ~10–50KB per document sparse vector vs ~1.5KB for dense — more expensive than dense but uses standard inverted index infrastructure

**General production concerns:**
- Model version drift applies to all of these: changing the SPLADE model version changes all document encodings, requiring full re-indexing
- SPLADE and ColBERT require GPU at index time — your RTX 4060 (8GB) is sufficient for both at your corpus scale
- None of these architectures are standard in Qdrant's built-in offerings — SPLADE sparse vectors can be stored in Qdrant's sparse vector support; ColBERT requires specialised infrastructure (`ragatouille`) or PLAID

---

### 8. Design Decisions and Trade-offs

**When to use DPR over symmetric bi-encoder:**
- When your queries and documents have clearly different linguistic styles (questions vs answers, emails vs policy clauses) — yes, this is your project
- When you have or can create labeled (query, positive passage) training pairs
- Without training pairs: use E5 with "query:" and "passage:" prefixes instead

**When to use ColBERT:**
- When single-vector retrieval quality is measurably insufficient (confirmed by evaluation, not assumption)
- When you have enough storage for the multi-vector representation (trivial at your scale)
- When you want the highest retrieval quality achievable without a cross-encoder
- For production: commit to `ragatouille` as the infrastructure

**When to use SPLADE:**
- When vocabulary mismatch is the dominant failure mode (it is, for your Hinglish corpus)
- When you already have BM25 infrastructure (Elasticsearch, OpenSearch) and want semantic expansion without changing to vector databases
- When interpretability matters: SPLADE's non-zero dimensions tell you exactly why a document was retrieved
- When you cannot afford the storage and complexity overhead of ColBERT

**When to use uniCOIL:**
- When SPLADE is too slow to index and too large for your infrastructure
- When you are willing to accept slightly lower quality for significantly faster indexing

**Do not use any of these when:**
- Hybrid BM25 + dense with RRF (Topic 4) has not been tried first — it is much simpler and often achieves 80–90% of the quality improvement at a fraction of the cost

---

### 9. Alternatives and When to Use Each

**E5 (`intfloat/multilingual-e5-large` or `-base`):**
- Drop-in upgrade over `paraphrase-multilingual-MiniLM-L12-v2`
- Uses "query:" and "passage:" prefixes at encoding time — asymmetric in the DPR spirit without custom training
- No training, no infrastructure change, just a model swap and full re-embed
- For your project: the first thing to try after measuring that the current model's 0.0393 gap is causing real retrieval failures

**GTE, BGE, Cohere embed v3:**
- Alternative strong multilingual embedding models
- BGE-M3 (FlagEmbedding) is particularly interesting: produces dense vectors, sparse vectors (SPLADE-like), and ColBERT-style token vectors simultaneously from one model
- Qdrant supports all three natively

**BGE-M3 for your project specifically:**
- One model, three retrieval modes: dense + sparse + multi-vector
- Could replace your current bi-encoder AND add SPLADE-like sparse retrieval AND add ColBERT-like token matching in one deployment
- 570M parameters vs 117M for MiniLM — ~5× larger, ~3–5× slower
- Qdrant has native support for BGE-M3's hybrid output

---

### 10. Common Mistakes and Production Failures

- **Using DPR off-the-shelf on a domain it was not trained on**: DPR was trained on Natural Questions (Wikipedia QA) — using it directly on FD policy retrieval without fine-tuning gives worse results than `paraphrase-multilingual-MiniLM-L12-v2` because the domain is completely different
- **Not using prefixes with E5**: `intfloat/multilingual-e5-large` requires "query: " prepended to queries and "passage: " prepended to passages during encoding — forgetting the prefix significantly degrades quality
- **Attempting ColBERT without `ragatouille`**: implementing MaxSim from scratch requires careful attention to the two-stage retrieval, compression, and decompression — `ragatouille` provides this already; don't reinvent it
- **Using SPLADE vocabulary expanded vectors in a standard vector database**: SPLADE produces sparse vectors over a 30,000-dim vocabulary, not dense 384-dim vectors — they cannot be indexed in a standard dense vector store; use Elasticsearch or Qdrant's sparse vector support
- **Switching from your current model to DPR/ColBERT/SPLADE before measuring baseline hybrid performance**: Topic 4 (Hybrid BM25 + dense + RRF) often achieves most of the quality improvement at zero training cost — always measure this first before investing in more complex architectures

---

### 11. Lead-Level Interview Questions

**Basic:**

**Q: What is the difference between a bi-encoder and a cross-encoder?**

A: A bi-encoder encodes query and document independently into separate vectors, then computes similarity between those vectors. It is fast because document vectors can be precomputed and indexed; query embedding is the only inference needed at query time. A cross-encoder takes query and document together as a joint input and outputs a single relevance score — much more accurate because the model sees direct query-document interactions, but cannot be used for large-scale retrieval since you cannot precompute scores for every (query, document) pair. In production: bi-encoder for retrieval (first-pass, full corpus), cross-encoder for reranking (second-pass, top-k candidates only).

**Q: Why does DPR use two encoders instead of one?**

A: Questions and answers have different linguistic structures. A question is interrogative, short, often incomplete. A passage is declarative, complete, answer-shaped. Training one encoder to produce close vectors for both question style and answer style simultaneously produces a compromise that is optimal for neither. Two separate encoders let each specialise: `E_Q` learns what question embeddings should look like, `E_P` learns what answer embeddings should look like, and contrastive training with hard negatives pulls matched pairs together.

**Intermediate:**

**Q: What is MaxSim in ColBERT, and why is it more expressive than cosine similarity between single vectors?**

A: MaxSim computes, for each query token, the maximum cosine similarity with any document token, then sums those maxima. For the query "premature withdrawal penalty", MaxSim independently finds the best match for "premature" (maybe "early" in a document), "withdrawal" (maybe "closure"), and "penalty" (maybe "charges"), then sums these. A single-vector approach blends all three concepts into one vector — the average may not be close to the document's average even if each concept individually matches well. MaxSim treats each query concept as an independent requirement that must match somewhere in the document, giving much finer-grained matching.

**Q: How does SPLADE bridge vocabulary mismatch, and why is it different from dense retrieval?**

A: SPLADE applies a transformer's MLM head to produce weights over the full vocabulary for each document. A document about "premature withdrawal penalty" gets non-zero weights not just for those exact words but for related terms the model learned to associate during training — "closure", "charges", "deduction", "forfeit", etc. This is vocabulary expansion: terms are added to the document's sparse representation even if they never appear in the text. Dense retrieval addresses vocabulary mismatch by mapping all texts to a shared geometric space where "premature withdrawal" and "early closure" are nearby — but this depends on the model having been trained to produce similar vectors for these phrases. SPLADE's expansion is more interpretable (you can see exactly which expanded terms explain a match) and uses inverted index infrastructure (no vector database needed at query time).

**Advanced:**

**Q: Your 64.4% Hinglish corpus has significant vocabulary mismatch between customer queries and English policy documents. You have 17 policy pages and 2,500 emails. Which of DPR, ColBERT, or SPLADE would you implement first, and why?**

A: None of them — first I would implement hybrid BM25 + dense with RRF (Topic 4), because it requires no training, no new infrastructure, and no re-embedding beyond what is already in Qdrant from Chapter 6. It directly addresses both the exact-match strength of BM25 and the vocabulary mismatch coverage of dense retrieval. This baseline often achieves 80–90% of the quality improvement of more sophisticated methods. After measuring hybrid retrieval's Recall@K on a labeled evaluation set, if vocabulary mismatch remains the primary failure mode (specifically the Hinglish-English gap), I would then evaluate SPLADE — because SPLADE's expansion can bridge "byaj" to "interest" and "machurity" to "maturity" at the sparse retrieval layer, without requiring custom training data (pre-trained multilingual SPLADE models exist). DPR requires labeled (query, passage) pairs that do not currently exist for this corpus. ColBERT is most valuable when fine-grained token-level matching is the bottleneck, which comes after vocabulary mismatch is addressed.

**Q: A teammate suggests replacing your current retrieval entirely with ColBERT because "it has token-level precision and will definitely be better." How do you respond?**

A: ColBERT is more expressive than single-vector retrieval, but "definitely better" requires measurement on your specific corpus. The risks of premature adoption: ColBERT requires specialised infrastructure (ragatouille or PLAID), significantly more storage per document (100 vectors per document vs 1), and the public ColBERT checkpoints were trained on English datasets. For a Hinglish corpus, a multilingual ColBERT model would need to be found or fine-tuned. More importantly, the current system's retrieval quality has not been measured systematically — we do not know if the bottleneck is actually in retrieval or in generation. The correct approach is: measure Recall@K with the current BM25 + dense hybrid system on a labeled evaluation set, identify the specific failure patterns, then choose the architecture that addresses those specific patterns. If token-level mismatch is confirmed as the primary failure, ColBERT is the right answer. If it is vocabulary mismatch, SPLADE addresses it with simpler infrastructure. If it is simply the thin 0.0393 gap from a weak embedding model, switching to E5 with "query:" and "passage:" prefixes is a free, zero-training fix.

**Scenario-based:**

**Q: After deploying a hybrid BM25 + dense system, you measure Recall@10 = 0.72 on your FD policy retrieval evaluation set. The most common failure pattern is that customer emails in Hinglish about FD maturity consistently fail to retrieve the maturity-related policy chunks. Walk through your diagnosis and the specific technical steps to fix this.**

A: Start by confirming the root cause. Sample the failed queries and look at the BM25 scores — if the Hinglish maturity queries score zero against all policy chunks, vocabulary mismatch is confirmed as the primary issue. Then look at the dense scores — if they are in the 0.40–0.45 range (close to your measured background noise), the embedding model is not bridging the Hinglish-English gap.

Fix options in order of implementation cost. First, query expansion: build a Hinglish synonym dictionary from the co-occurrence patterns in your 2,500 emails — "machurity" → "maturity", "byaj" → "interest", "paisa aaya" → "maturity proceeds credited". Prepend expanded terms to queries before BM25 lookup. This is one day of work and can be done without any infrastructure change. Second, upgrade the embedding model: switch from `paraphrase-multilingual-MiniLM-L12-v2` to `intfloat/multilingual-e5-large` with "query:" and "passage:" prefixes. Re-embed the 17 pages and update Qdrant. Measure Recall@10 again — if it improves significantly, the embedding model was the bottleneck. Third, if both of the above are insufficient, evaluate a pre-trained multilingual SPLADE model on the same evaluation set. SPLADE's vocabulary expansion may learn to bridge the Hinglish-English gap without any fine-tuning if trained on multilingual retrieval pairs. Fourth, if fine-tuning budget exists, create (Hinglish FD email, English policy chunk) training pairs by having domain experts label 200–500 pairs, then fine-tune DPR or SPLADE on those pairs specifically. Expect the biggest quality jump from this but at the highest cost.

---

### 12. Hidden Concepts and Prerequisites

**Contrastive learning as the training foundation:**
- DPR, ColBERT, and SPLADE all use variants of contrastive learning: push together representations of matching pairs, push apart non-matching pairs
- The key hyperparameter is the temperature in the softmax: lower temperature makes the loss focus more on hard negatives (the "most wrong" negatives), higher temperature distributes the gradient more evenly
- Hard negative mining (using BM25 to find "almost-good-but-wrong" passages) is what separates good from great retrieval models — easy negatives teach nothing

**In-batch negatives:**
- A training efficiency technique: in a batch of N (query, passage) pairs, each query treats the other N-1 passages as negatives without needing explicit negative labels
- Effective batch size for in-batch negatives is the bottleneck — DPR used 4×8 GPU batches with 128 questions each, making effective batch size very large
- For fine-tuning on your corpus: limited by GPU VRAM (your RTX 4060 has 8GB); batch size of 8–16 with gradient accumulation is a realistic starting point

**Knowledge distillation from cross-encoders:**
- Many production retrieval models (SPLADE v2, ColBERT v2) use a cross-encoder as a "teacher" to generate soft labels for training the bi-encoder "student"
- The cross-encoder evaluates every (query, passage) pair in the training data and produces a relevance score; the bi-encoder is then trained to match these soft scores rather than binary 0/1 labels
- This is more informative than binary labels and significantly improves retrieval quality without needing human-labeled data beyond the initial cross-encoder training

**MaxSim is a special case of a more general interaction pattern:**
- BERT's cross-attention in cross-encoders is full interaction: every query token attends to every document token bidirectionally
- ColBERT's MaxSim is late interaction: token vectors are computed independently, then interact at scoring time
- Bi-encoders (DPR, dense retrieval) are no interaction: both sides are fully independent until the final dot product
- The interaction → independence tradeoff maps directly to quality → speed: more interaction = higher quality = lower throughput

---

### 13. Revision Summary

> Four architectures extend beyond basic bi-encoder dense retrieval:
>
> **DPR** — two separate encoders (query tower + passage tower) trained on (question, positive, negative) triplets. Better when query and document styles differ. Requires labeled training pairs. Practical upgrade: use E5 with "query:" and "passage:" prefixes for free asymmetric encoding without training.
>
> **ColBERT** — one vector per token, MaxSim scoring (sum of max cosine per query token). Much more expressive than single-vector but requires 100× more storage per document. Available via `ragatouille`. Best for high-quality retrieval when fine-grained token-level matching matters.
>
> **SPLADE** — transformer-based vocabulary expansion produces learned sparse vectors over the full 30,000+ term vocabulary. Bridges vocabulary mismatch at the sparse retrieval layer. Uses inverted index infrastructure at query time (millisecond latency, no vector DB needed). Most relevant for your Hinglish corpus where "byaj" needs to match "interest" in policy documents.
>
> **Learned sparse (uniCOIL, BGE-M3)** — simplified variants: uniCOIL produces one weight per input token (not full vocabulary expansion); BGE-M3 produces dense + sparse + multi-vector simultaneously from one model.
>
> **For your project:** implement hybrid BM25 + dense + RRF (Topic 4) first. If Recall@K remains insufficient, upgrade the embedding model to E5 (free). Then evaluate SPLADE for the Hinglish vocabulary mismatch problem. ColBERT is the ceiling for retrieval quality without a cross-encoder. Always measure before choosing — the 0.0393 discrimination gap is real but the correct fix depends on which failure mode dominates after measuring.

---
